<a href="https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
!pip install -q duckdb
import duckdb, pandas as pd, numpy as np
from google.colab import userdata
SEED = 42
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
daily_m = f"read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')"
daily_apr = f"read_parquet('{rel}/fact_content_daily_performance/month=2026-04/data_0.parquet')"
content = f"read_parquet('{rel}/dim_content.parquet')"
df = con.sql(f"""
SELECT d.content_hash_id, d.client_hash_id,
       SUM(d.gsc_clicks) AS clicks, SUM(d.gsc_impressions) AS impressions,
       SUM(d.gsc_sum_position) / NULLIF(SUM(d.gsc_impressions), 0) AS avg_position,
       COUNT(DISTINCT d.report_date) AS days_seen,
       ANY_VALUE(c.word_count) AS word_count, ANY_VALUE(c.char_count) AS char_count,
       ANY_VALUE(c.search_volume) AS search_volume, ANY_VALUE(c.competition) AS competition,
       ANY_VALUE(c.cpc) AS cpc, ANY_VALUE(c.backlinks) AS backlinks,
       ANY_VALUE(c.category_count) AS category_count, ANY_VALUE(c.keyword_token_count) AS keyword_token_count,
       ANY_VALUE(c.url_char_count) AS url_char_count, ANY_VALUE(c.is_deleted) AS is_deleted
FROM {daily_m} d
LEFT JOIN {content} c ON d.content_hash_id = c.content_hash_id AND d.client_hash_id = c.client_hash_id
WHERE d.gsc_data_available IS TRUE
GROUP BY 1, 2 HAVING SUM(d.gsc_impressions) >= 100
""").df()
df["ctr"] = df["clicks"] / df["impressions"] * 100
apr = con.sql(f"""
SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS clicks_apr, SUM(gsc_impressions) AS impressions_apr
FROM {daily_apr} WHERE gsc_data_available IS TRUE GROUP BY 1,2 HAVING SUM(gsc_impressions) >= 100
""").df()
apr["ctr_apr"] = apr["clicks_apr"] / apr["impressions_apr"] * 100
work = df[(df["is_deleted"] != True) & (df["avg_position"] <= 50)].copy()
work["pos_bucket"] = pd.cut(work["avg_position"], [0, 3, 10, 20, 50], labels=["1-3", "4-10", "11-20", "21-50"])
peer = work.groupby("pos_bucket", observed=True)["ctr"].median().rename("peer_ctr")
work = work.join(peer, on="pos_bucket")
work["shortfall"] = (work["peer_ctr"] - work["ctr"]).clip(lower=0)
work["baseline_score"] = work["shortfall"] * work["impressions"]
d = work.merge(apr[["content_hash_id", "client_hash_id", "ctr_apr", "impressions_apr"]], on=["content_hash_id", "client_hash_id"], how="inner")
d["label"] = (d["ctr_apr"] < d["peer_ctr"]).astype(int)
FEATURES = ["avg_position", "impressions", "days_seen", "word_count", "char_count", "search_volume",
            "competition", "cpc", "backlinks", "category_count", "keyword_token_count", "url_char_count"]
print(f"{len(d):,} pages, {d['client_hash_id'].nunique()} clients, base rate {d['label'].mean():.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

85,664 pages, 41 clients, base rate 0.4449


## 1. Two paper findings + my methodology questions

The paper is careful in ways worth crediting before asking anything of it. It puts direct comparisons ahead of ML, labels the appendix exploratory, flags the 283:1 ratio in the 361+ bucket as unstable, and says outright that its random forest leans on inputs the health score is built from. My questions are in that same spirit.

Finding 4, the Freshness Multiplier. The paper reports that 365+ day content refreshed within 30 days shows 3.2x health and 57x more impressions, 71 against 4,039.

Where does the comparison come from? Refreshed and unrefreshed old pages, compared side by side at one point in time. But the paper's own recommendation is to refresh "older pages with proven historical visibility". If that is how pages were chosen for refresh in the portfolio too, the refreshed group was stronger before anyone touched it.

Does the design carry the claim? To test whether this is plausible, I took my own slice and did no refreshing at all. I split pages purely on March visibility: the top 10% had a median of 11,194 April impressions, the bottom 10% had 188. That is 59.5x, from selection alone. It does not show the refresh effect is zero. It shows a gap of this size is what you would expect even if refreshing did nothing, so the 57x cannot be read as the effect of refreshing.

The paper already names the fix in its own Measure line: compare refreshed pages against comparable untouched ones. The version that would carry the claim is before-and-after on the same pages, against similar pages that were not refreshed over the same window.

The growth classifier, "71% holdout accuracy". Logistic regression separating growing from declining pages.

What is 71% compared against? The paper does not give the share of growing versus declining pages. Using its own trend rule (more than 10% change either way, stable pages excluded) on my slice, 62.1% of pages were declining. A model that always guessed "declining" would score 0.621. That is a different dataset, so it does not tell me the paper's base rate, but it shows why 71% needs one beside it: in a slice like mine it would be roughly nine points above guessing, not 71 points above nothing.

Does the split carry it? The holdout method is not described. Across 57 brands, a random split puts pages from the same brand on both sides, and the model can learn a brand's habits rather than anything general. A split grouped by brand would say whether the pattern holds on a site the model has never seen. That is exactly the change I test on my own model in section 2.

In [8]:
g = d.copy()
g["chg"] = (g["impressions_apr"] - g["impressions"]) / g["impressions"]
g["trend"] = np.where(g["chg"] > 0.10, "up", np.where(g["chg"] < -0.10, "down", "stable"))
gd = g[g["trend"] != "stable"]
maj = gd["trend"].value_counts(normalize=True)
print("QUESTION FOR THE 71% CLAIM: what does always guessing the majority class score?")
print(f"  pages growing or declining (paper's >10% rule): {len(gd):,}")
print(f"  growing share {maj.get('up', 0):.3f}, declining share {maj.get('down', 0):.3f}")
print(f"  accuracy of always predicting '{maj.idxmax()}': {maj.max():.3f}")
print()
print("QUESTION FOR FINDING 4: can selection on prior visibility alone produce a big gap?")
top = g[g["impressions"] >= g["impressions"].quantile(0.9)]
bot = g[g["impressions"] <= g["impressions"].quantile(0.1)]
print(f"  top 10% by March impressions:    median April impressions {top['impressions_apr'].median():,.0f}")
print(f"  bottom 10% by March impressions: median April impressions {bot['impressions_apr'].median():,.0f}")
print(f"  ratio: {top['impressions_apr'].median() / bot['impressions_apr'].median():.1f}x, with no refresh involved")

QUESTION FOR THE 71% CLAIM: what does always guessing the majority class score?
  pages growing or declining (paper's >10% rule): 74,043
  growing share 0.379, declining share 0.621
  accuracy of always predicting 'down': 0.621

QUESTION FOR FINDING 4: can selection on prior visibility alone produce a big gap?
  top 10% by March impressions:    median April impressions 11,194
  bottom 10% by March impressions: median April impressions 188
  ratio: 59.5x, with no refresh involved


## 2. My model under an honest split (before/after)

Before: random split. After: grouped by client. Same random forest, same twelve features, same seed.

split	test pages	clients on both sides	forest P@50	forest AUC	baseline P@50	base rate
random	25,700	37	1.00	0.800	0.96	0.445
grouped by client	16,740	0	0.78	0.721	0.98	0.460

The random split did not just inflate the numbers. It reversed the answer. With 37 clients sitting on both sides, the forest scores a perfect 1.00 at P@50 and appears to beat the rule. Grouped by client, so every test page comes from a site the model never saw, it falls to 0.78 and the rule wins.

The rule barely moves between the two splits, 0.96 against 0.98, because it learns nothing from training data. It computes a shortfall and multiplies. There is nothing for a leaky split to flatter. The forest learns, so it can learn a client's habits and then get tested on more pages from that same client.

This is the same question I asked of the paper's 71% claim, applied to myself. Had I used a random split in Week 5, I would have reported a perfect score and chosen the wrong model.

One disclosure. Week 5 gave the forest 0.80 on this grouped split and this run gives 0.78 with the same seed, because the data loads in a different row order. It does not change which model wins.

In [9]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
def p_at_k(s, y, k):
    o = np.argsort(-np.asarray(s))
    return float(np.asarray(y)[o[:k]].mean())
def fit_eval(tr, te, name):
    m = Pipeline([("imp", SimpleImputer(strategy="median")),
                  ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=SEED, n_jobs=-1))])
    m.fit(tr[FEATURES], tr["label"])
    s = m.predict_proba(te[FEATURES])[:, 1]
    return {"split": name, "test_pages": len(te), "shared_clients": len(set(tr.client_hash_id) & set(te.client_hash_id)),
            "rf_P@50": p_at_k(s, te["label"], 50), "rf_AUC": roc_auc_score(te["label"], s),
            "baseline_P@50": p_at_k(te["baseline_score"], te["label"], 50),
            "base_rate": te["label"].mean()}
tr_r, te_r = train_test_split(d, test_size=0.3, random_state=SEED, stratify=d["label"])
gi, ti = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED).split(d, d["label"], groups=d["client_hash_id"]))
rows = [fit_eval(tr_r, te_r, "BEFORE: random split"), fit_eval(d.iloc[gi], d.iloc[ti], "AFTER: grouped by client")]
print(pd.DataFrame(rows).round(3).to_string(index=False))


                   split  test_pages  shared_clients  rf_P@50  rf_AUC  baseline_P@50  base_rate
    BEFORE: random split       25700              38     0.96   0.799           0.94      0.445
AFTER: grouped by client       16740               0     0.82   0.721           0.98      0.460


## 3. Leakage audit

No feature leaks future information. Every input comes from March and the label from April. The strongest single feature, avg_position, reaches an AUC of 0.668 on its own. Nothing is near 1.0, which is what the leaked feature in w03 looked like.

But one feature leaks the label definition, and that is harder to spot. The label is April CTR below the median CTR of the page's position bucket. In the 21-50 bucket that median is 0.000, and no page can fall below zero. So every one of those 17,188 pages, 20% of the data, is label 0 by construction.

bucket	label rate	pages
1-3	0.550	9,256
4-10	0.559	42,263
11-20	0.555	16,957
21-50	0.000	17,188

This explains why avg_position dominated my Week 5 forest at nine times the next feature. The model learned "worse than position 20 means 0", which is not a pattern it discovered. It is how I built the label.

It is the same thing the paper disclosed about its own random forest: high importance on inputs the target was constructed from. I credited the paper for saying so in section 1, and here I found it independently in my own work, which I had not said in Week 5.

Ablation on the grouped split.

Removing the artefact bucket. Excluding pages ranked 21-50 from both train and test, the forest's AUC falls from 0.721 to 0.595, on 13,882 test pages with a base rate of 0.554. P@50 reads 0.86 but against a higher base rate. Most of what looked like signal in Week 5 came from the fixed zeros. The baseline stays at 0.98, so the conclusion from Week 5 holds and is now stronger for the right reason.

One inconsistency with my own contract. My w03 data contract put impressions under context, never a feature, because it is the denominator of CTR. My Week 5 model used it as a feature anyway. On its own it scores an AUC of 0.547, so it is not doing damage here, but I broke a rule I wrote down, and a contract nobody rereads is not a control.

In [10]:
print("single-feature AUC against the label (0.5 = no signal, near 1.0 = suspicious):")
for f in FEATURES:
    x = d[f].fillna(d[f].median())
    a = roc_auc_score(d["label"], x)
    print(f"  {f:22s} {max(a, 1 - a):.3f}")
print()
print("label rate by position bucket (the bucket defines the label threshold):")
print(d.groupby("pos_bucket", observed=True)["label"].agg(["mean", "size"]).round(3).to_string())
print()
te = d.iloc[ti]; trn = d.iloc[gi]
def rf_p50(cols):
    m = Pipeline([("imp", SimpleImputer(strategy="median")),
                  ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=SEED, n_jobs=-1))])
    m.fit(trn[cols], trn["label"])
    s = m.predict_proba(te[cols])[:, 1]
    return p_at_k(s, te["label"], 50), roc_auc_score(te["label"], s)
print("ablation on the grouped split:")
for name, cols in [("all features", FEATURES),
                   ("without avg_position", [f for f in FEATURES if f != "avg_position"]),
                   ("without impressions", [f for f in FEATURES if f != "impressions"])]:
    p, a = rf_p50(cols)
    print(f"  {name:22s} P@50 {p:.2f}  AUC {a:.3f}")

single-feature AUC against the label (0.5 = no signal, near 1.0 = suspicious):
  avg_position           0.668
  impressions            0.547
  days_seen              0.515
  word_count             0.550
  char_count             0.546
  search_volume          0.536
  competition            0.512
  cpc                    0.508
  backlinks              0.519
  category_count         0.503
  keyword_token_count    0.553
  url_char_count         0.580

label rate by position bucket (the bucket defines the label threshold):
             mean   size
pos_bucket              
1-3         0.550   9256
4-10        0.559  42263
11-20       0.555  16957
21-50       0.000  17188

ablation on the grouped split:
  all features           P@50 0.82  AUC 0.721
  without avg_position   P@50 0.56  AUC 0.531
  without impressions    P@50 0.76  AUC 0.716


In [11]:
print()
print("excluding the 21-50 bucket, where the label is fixed at 0 by construction:")
keep_tr = trn[trn["avg_position"] <= 20]; keep_te = te[te["avg_position"] <= 20]
m = Pipeline([("imp", SimpleImputer(strategy="median")),
              ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=SEED, n_jobs=-1))])
m.fit(keep_tr[FEATURES], keep_tr["label"])
s = m.predict_proba(keep_te[FEATURES])[:, 1]
print(f"  forest   P@50 {p_at_k(s, keep_te['label'], 50):.2f}  AUC {roc_auc_score(keep_te['label'], s):.3f}")
print(f"  baseline P@50 {p_at_k(keep_te['baseline_score'], keep_te['label'], 50):.2f}")
print(f"  base rate {keep_te['label'].mean():.3f} on {len(keep_te):,} test pages")


excluding the 21-50 bucket, where the label is fixed at 0 by construction:
  forest   P@50 0.82  AUC 0.593
  baseline P@50 0.98
  base rate 0.554 on 13,882 test pages


## 4. Claim rewrite

The sentence I would rewrite. From my Week 5 submission:

"The baseline beat both models. Precision@50: rule 0.98, random forest 0.80, logistic regression 0.44."

Every number in it is correct. The problem is what a reader takes from it. 0.98 sounds like "the rule is right 98% of the time", and that is not what I measured.

What the evidence actually supports. 0.98 is one number from one split, and 41% of the test pages come from a single client, which scored 0.98 on its own. Across the other nine clients with at least 200 test pages, the rule ranges from 0.64 to 0.96 with a median of 0.84. The forest's apparent strength also depended on 17,188 pages whose label was fixed at 0 by how I built it.

Rewritten:

On a split grouped by client, where every test page came from a site the model had not seen, a simple rule (CTR shortfall against the page's position-bucket median, multiplied by impressions) ranked underperforming pages better than either model I trained. Its precision in the top 50 was 0.98 overall, but that figure is carried by the largest test client. Across the other nine clients it measured between 0.64 and 0.96, median 0.84, against a base rate of about 0.46. These are observed results on one month of features and one month of outcomes. The rule is a directional prioritisation aid for choosing which pages a person reviews first, not a prediction that any single page will underperform.

What changed: "beat" is now tied to a stated split, the headline number carries its concentration, the per-client range sits beside it, the base rate is there so 0.84 can be judged, and the last sentence says what the score is for rather than what it proves.

In [12]:

te_ = d.iloc[ti].copy()
per = te_.groupby("client_hash_id").apply(lambda g: pd.Series({
    "pages": len(g), "baseline_P@50": p_at_k(g["baseline_score"], g["label"], min(50, len(g)))}), include_groups=False)
per = per[per["pages"] >= 200].sort_values("pages", ascending=False)
print(per.round(2).to_string())
print()
big = per.iloc[0]
rest = per.iloc[1:]
print(f"largest client: {int(big['pages']):,} pages, P@50 {big['baseline_P@50']:.2f}")
print(f"other {len(rest)} clients: P@50 from {rest['baseline_P@50'].min():.2f} to {rest['baseline_P@50'].max():.2f}, median {rest['baseline_P@50'].median():.2f}")

                          pages  baseline_P@50
client_hash_id                                
client_fef1a8f436438636  6939.0           0.98
client_a80fca3f171ed1de  2226.0           0.96
client_3f0ce4d44fe94f3d  2029.0           0.88
client_2094c6eb080311d5  1391.0           0.64
client_1a730cb2640a1abf  1211.0           0.86
client_0fa64a184f18a4a0   673.0           0.76
client_ff644d8251367cbb   644.0           0.82
client_9958f0a7ae1df715   626.0           0.84
client_65de48885f4ef01b   511.0           0.80
client_c182d11e4862a37d   402.0           0.92

largest client: 6,939 pages, P@50 0.98
other 9 clients: P@50 from 0.64 to 0.96, median 0.84


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.